# SIC walkthrough: DeepHit10

An end-to-end tour of the `sic` package with the **DeepHit10** head:

1. Load the model and inspect the **raw** head outputs, then read off the survival probabilities at the 10 native time points.
2. Predict on a **custom `eval_times` grid**, with and without **post-training calibration** (CiPOT), and plot the curves.
3. Compute the **full metric suite** with the `sic.metrics` module.
4. **Cross-validate over every dataset**, save the per-fold metrics to CSV, and report `mean +/- std` (C-index, D-calibration, IBS) read back from that CSV.

Requirements: `pip install sic` (plus `matplotlib` for the plots). Model weights and datasets download automatically from [yayapa/sic](https://huggingface.co/yayapa/sic) on first use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


from sic import SIC, make_y, load_dataset, list_datasets

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
RANDOM_STATE = 42

## 1. Load the model and inspect raw outputs

We use the METABRIC breast-cancer cohort. The `SIC` model downloads the DeepHit10 checkpoint from the Hub on first use. There is no gradient training at fit time: the training rows are consumed as in-context examples in a single forward pass.

In [ ]:
data = load_dataset("GBSG")
x, t, e = data["x"], data["t"], data["e"]
num_cols = data["num_cols"]

x_tr, x_te, t_tr, t_te, e_tr, e_te = train_test_split(
    x, t, e, test_size=0.3, random_state=RANDOM_STATE
)

# Standardize numeric covariates on the training fold (SIC's own standardize=False here).
scaler = StandardScaler()
x_tr = x_tr.copy(); x_te = x_te.copy()
x_tr[num_cols] = scaler.fit_transform(x_tr[num_cols])
x_te[num_cols] = scaler.transform(x_te[num_cols])

y_tr = make_y(t_tr, e_tr)

model = SIC(device="cuda")
print(f"device={model.device}  head={model.survival_head}")
print(f"train={x_tr.shape}  test={x_te.shape}  event rate={e_tr.mean():.3f}")

In [ ]:
# Raw head outputs: for DeepHit10 that is 10 logits + 10 PMF values per subject,
# plus the 10 discrete-time bin edges shared across subjects.
raw = model.fit_predict_raw(x_tr, y_tr, x_te)
print("raw keys:", list(raw.keys()))
print("logits:", raw["logits"].shape, "| pmf:", raw["pmf"].shape, "| bin_edges:", raw["bin_edges"].shape)
print("\nbin edges (10 native time points):")
print(np.round(raw["bin_edges"], 2))

In [ ]:
# Survival at the 10 native time points = 1 - cumulative PMF (per subject).
# Equivalently, ask the model for its native grid (eval_times=None).
surv_native = model.fit_predict_surv(x_tr, y_tr, x_te)  # index = 10 bin edges
print("native survival curve shape (times, subjects):", surv_native.shape)
print("\nSurvival at the 10 native time points, first 5 test subjects:")
surv_native.iloc[:, :5].round(3)
# print the survival curve for them using matplotlib
plt.plot(surv_native.index, surv_native.iloc[:, :5])
plt.show()

## 2. Custom `eval_times` + post-training calibration

Now we evaluate on an arbitrary time grid (not the native bins). We predict both the plain curves and the CiPOT-calibrated curves, then overlay them for a few subjects.

In [ ]:
# Dense grid over the observed event times for a smooth curve.
eval_times = np.linspace(t_tr[e_tr == 1].min(), np.quantile(t_tr, 0.95), 50).tolist()

surv_plain = model.fit_predict_surv(x_tr, y_tr, x_te, eval_times=eval_times)
surv_cal = model.fit_predict_surv(
    x_tr, y_tr, x_te, eval_times=eval_times, post_training_calibration="cipot"
)

print("plain:", surv_plain.shape, "| calibrated:", surv_cal.shape)
surv_cal.iloc[:, :5].round(3).head()

In [ ]:
subjects = [0, 1, 2, 3]
fig, ax = plt.subplots(figsize=(8, 5))
for j in subjects:
    color = f"C{j}"
    ax.step(surv_plain.index, surv_plain.iloc[:, j], where="post",
            color=color, alpha=0.5, label=f"subj {j} (plain)")
    ax.step(surv_cal.index, surv_cal.iloc[:, j], where="post",
            color=color, linestyle="--", label=f"subj {j} (CiPOT)")
ax.set_xlabel("time"); ax.set_ylabel("survival probability")
ax.set_title("DeepHit10 survival curves: plain vs CiPOT-calibrated")
ax.set_ylim(0, 1.02); ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. The metrics module

All evaluation lives in `sic.metrics`. `evaluate_survival` returns the full suite (time-dependent C-index, IBS, td-AUC, D-calibration, MAE-PO, and per-quantile AUC/BS/C-index); `metrics_to_row` flattens it into one record.

In [ ]:
from sic.metrics import evaluate_survival, metrics_to_row

res_plain = evaluate_survival(surv_plain, t_te, e_te, t_tr, e_tr)
res_cal = evaluate_survival(surv_cal, t_te, e_te, t_tr, e_tr)

comparison = pd.DataFrame(
    {"plain": metrics_to_row(res_plain), "CiPOT": metrics_to_row(res_cal)}
)
comparison

## 4. Cross-validate over all datasets and save a CSV

`sic.metrics.cross_validate` runs k-fold CV of the model across a list of datasets, standardizing per fold, and returns one row per (dataset, fold). We save it to CSV, then read it back and report `mean +/- std`.

> This loops over every bundled/auto-downloaded dataset and can take a while on CPU. Trim `DATASETS` or `k_folds` for a quick run.

In [ ]:
from sic.metrics import cross_validate
# note download of METABRIC and GBSG from deepsurv repo may take a while
CSV_PATH = "sic_deephit10_cv_results.csv"
DATASETS = list_datasets()          # every supported dataset
# For a fast demo instead, use e.g.:
#DATASETS = ["VETERAN", "LUNG", "PBC", "WHAS500", "GBSG2"]

results = cross_validate(
    model,
    DATASETS,
    k_folds=5,
    seed=RANDOM_STATE,
    post_training_calibration=None,     # set "cipot" / "csd" to evaluate calibrated curves
    csv_path=CSV_PATH,
)
print(f"\nsaved {len(results)} rows to {CSV_PATH}")
results.head()

In [ ]:
# Read the saved CSV back and summarize the headline metrics as mean +/- std.
from sic.metrics import summarize

df = pd.read_csv(CSV_PATH)
summary = summarize(df, metrics=("cindex_td", "d_cal", "ibs"))
summary[["cindex_td", "d_cal", "ibs"]].rename(
    columns={"cindex_td": "C-index (td)", "d_cal": "D-calibration", "ibs": "IBS"}
)

In [ ]:
# Bar chart of C-index across datasets with std error bars.
order = summary["cindex_td_mean"].sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(order, summary.loc[order, "cindex_td_mean"],
       yerr=summary.loc[order, "cindex_td_std"], capsize=3, color="C0", alpha=0.85)
ax.axhline(0.5, color="k", ls="--", lw=1, label="random (0.5)")
ax.set_ylabel("C-index (td)"); ax.set_ylim(0.4, 1.0)
ax.set_title("DeepHit10 discrimination across datasets (mean +/- std over folds)")
ax.tick_params(axis="x", rotation=45); ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()